# ActiveMQ Artemis Replication HA (Live/Backup) with Docker Compose

## Goal
- **2 Artemis brokers**
- **Replication HA** (no shared disk)
- **One live (master)**, **one backup (slave)**
- Clients reconnect via **failover URL**


## 1) Project layout

Create this folder structure:

```
artemis-repl-ha/
  docker-compose.yml
  conf/
    live/
      broker.xml
    backup/
      broker.xml
```


## 2) `docker-compose.yml`

In [ ]:
docker_compose_yml = r'''version: "3.8"

services:
  artemis-live:
    image: apache/activemq-artemis:latest
    container_name: artemis-live
    hostname: artemis-live
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
    volumes:
      - artemis-live-data:/var/lib/artemis-instance
      - ./conf/live/broker.xml:/var/lib/artemis-instance/etc/broker.xml:ro
    ports:
      - "61616:61616"  # core JMS
      - "8161:8161"    # web console
    networks:
      - artemis-net

  artemis-backup:
    image: apache/activemq-artemis:latest
    container_name: artemis-backup
    hostname: artemis-backup
    depends_on:
      - artemis-live
    environment:
      ARTEMIS_USER: admin
      ARTEMIS_PASSWORD: admin
      ANONYMOUS_LOGIN: "false"
    volumes:
      - artemis-backup-data:/var/lib/artemis-instance
      - ./conf/backup/broker.xml:/var/lib/artemis-instance/etc/broker.xml:ro
    ports:
      - "61617:61616"  # core JMS (mapped differently on host)
      - "8162:8161"    # web console (mapped differently on host)
    networks:
      - artemis-net

volumes:
  artemis-live-data:
  artemis-backup-data:

networks:
  artemis-net:
'''
print(docker_compose_yml)


## 3) `conf/live/broker.xml` (LIVE / master)

This snippet focuses on the HA-related sections:
- `<connectors>`
- `<cluster-connections>`
- `<ha-policy>`


In [ ]:
live_broker_xml = r'''<?xml version='1.0'?>
<configuration xmlns="urn:activemq"
               xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:schemaLocation="urn:activemq /schema/artemis-configuration.xsd">

  <!--
    NOTE:
    This file shows only the relevant structure for replication HA.
    In real setups you typically start from the default broker.xml and
    *merge* these blocks into it.
  -->

  <core xmlns="urn:activemq:core">

    <name>artemis-live</name>

    <!-- 1) Connectors: both live and backup endpoints -->
    <connectors>
      <connector name="live-connector">tcp://artemis-live:61616</connector>
      <connector name="backup-connector">tcp://artemis-backup:61616</connector>
    </connectors>

    <!-- 2) Cluster connection: used by replication -->
    <cluster-connections>
      <cluster-connection name="my-cluster">
        <connector-ref>live-connector</connector-ref>
        <retry-interval>1000</retry-interval>
        <use-duplicate-detection>true</use-duplicate-detection>
        <message-load-balancing>ON_DEMAND</message-load-balancing>

        <static-connectors>
          <connector-ref>backup-connector</connector-ref>
        </static-connectors>
      </cluster-connection>
    </cluster-connections>

    <!-- 3) HA policy: replication master -->
    <ha-policy>
      <replication>
        <master>
          <check-for-live-server>true</check-for-live-server>
        </master>
      </replication>
    </ha-policy>

  </core>
</configuration>
'''
print(live_broker_xml)


## 4) `conf/backup/broker.xml` (BACKUP / slave)

Save the following as `conf/backup/broker.xml`.


In [ ]:
backup_broker_xml = r'''<?xml version='1.0'?>
<configuration xmlns="urn:activemq"
               xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xsi:schemaLocation="urn:activemq /schema/artemis-configuration.xsd">

  <core xmlns="urn:activemq:core">

    <name>artemis-backup</name>

    <!-- 1) Connectors: both live and backup endpoints -->
    <connectors>
      <connector name="live-connector">tcp://artemis-live:61616</connector>
      <connector name="backup-connector">tcp://artemis-backup:61616</connector>
    </connectors>

    <!-- 2) Cluster connection: points to live -->
    <cluster-connections>
      <cluster-connection name="my-cluster">
        <connector-ref>backup-connector</connector-ref>
        <retry-interval>1000</retry-interval>
        <use-duplicate-detection>true</use-duplicate-detection>
        <message-load-balancing>ON_DEMAND</message-load-balancing>

        <static-connectors>
          <connector-ref>live-connector</connector-ref>
        </static-connectors>
      </cluster-connection>
    </cluster-connections>

    <!-- 3) HA policy: replication slave -->
    <ha-policy>
      <replication>
        <slave>
          <allow-failback>true</allow-failback>
        </slave>
      </replication>
    </ha-policy>

  </core>
</configuration>
'''
print(backup_broker_xml)


## 5) Start the cluster

From inside the `artemis-repl-ha/` directory:

```bash
docker compose up -d
docker logs -f artemis-live
docker logs -f artemis-backup
```

Expected:
- `artemis-live` starts as **active**
- `artemis-backup` stays **passive**, replicating from live


## 6) Test failover

Stop the live broker:

```bash
docker stop artemis-live
docker logs -f artemis-backup
```

Expected:
- backup transitions to **active**


## 7) Client failover URL

Your clients should connect using a **failover URL**.

### From your host machine
Because we mapped ports differently:
- live: `localhost:61616`
- backup: `localhost:61617`

**Core JMS failover URL:**

```
failover:(tcp://localhost:61616,tcp://localhost:61617)
```

### From another container on the same Compose network
Use the internal service names:

```
failover:(tcp://artemis-live:61616,tcp://artemis-backup:61616)
```


## 8) Notes / gotchas

- Replication HA relies on a correct **cluster connection** (connectors + cluster-connections).
- Each broker must have its **own data directory** (we used two separate Docker volumes).
- In a real deployment, start from the default `broker.xml` shipped with the image and **merge** these HA blocks into it.
  The minimal XML shown here is meant for learning and quick prototypes.
